# Why default `ec_m` priors aren't enough

**Objective.** Meridian's out-of-the-box `ec_m` (half-saturation) prior is
channel-agnostic and weakly informative by design:
`ec_m ~ TruncatedNormal(0.8, 0.8, [0.1, 10])` -- the same distribution for
every channel, regardless of what's actually known about that channel's
audience size (see `meridian/model/prior_distribution.py`). This notebook
focuses on a single, narrow hypothesis, isolated from every other prior:

> On data with **known ground truth**, does a **reach-based `ec_m` prior**
> (derived from audience-size/frequency planning assumptions, not from the
> fitted model) recover the true half-saturation point substantially better
> than Meridian's default `ec_m` prior -- all else (including `alpha_m`'s
> prior) held at Meridian's own default?

This is a deliberately narrowed-down companion to
`meridian_priors_case_study.ipynb` (which also covers `alpha_m` and the
`ec_m`+`alpha_m` combination) -- here we only compare `default` vs.
`ec_only`, to keep the story to exactly the one hypothesis under test.

**The data-generating process.** `data_simulator.GeoMediaDataSimulator`
builds geo x time data in one linear pipeline: real state populations ->
per-geo audience-size heterogeneity -> reach x frequency media execution
(a shared annual seasonal signal + per-channel on/off flighting + per-channel
AR(1) noise) -> adstock + Hill saturation transforms (with `ec_m` derived
from a "half of target audience reached at the typical frequency"
assumption) -> a KPI generated from calibrated per-channel ROI targets.

**A caveat on "true half-saturation."** The 50%-of-audience threshold in
`simulate_adstock_hill_params()` is a stylized modeling choice for
generating a *known, self-consistent* ground truth -- not a claim that
real-world half-saturation empirically sits at 50% reach for any actual
channel. Everywhere this notebook says "true `ec_m`," it means "true under
this simulation's own stated assumption," not an independently validated
real-world threshold. The `ec_only` prior recovers that assumption because
it's built from the same audience/frequency inputs, which is the point
being tested here -- not evidence that 50% itself is the right number for
a real campaign. See `meridian_tv_underreach_case_study.ipynb` for a
scenario built on this same assumption, but with a much larger gap between
current execution and that threshold.

**How to read this notebook:**
- Section 1 builds one simulated dataset and shows its ground truth.
- Section 2 fits that single dataset under `default` vs. `ec_only` priors.
- Section 3 repeats the `default` vs. `ec_only` comparison across many
  random seeds and reports 90% HDI coverage for `ec_m` (and `roi_m`, since
  ROI is a downstream function of `ec_m`) -- turning the single-seed story
  into a calibration claim.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meridian.model import model

from data_simulator import SimulationConfig
from model_utils import build_model_spec
from model_utils import build_simulated_input
from model_utils import filter_param_summary

## 1. Build one simulated dataset and inspect its ground truth

In [ ]:
# Same planning-style overrides used in the companion notebook: TV strongly
# under-saturated, Display close to saturated, Social over-saturated;
# distinct target ROIs.
BASE_CONFIG_OVERRIDES = {
    'target_audience_pop_frac': {'TV': 0.50, 'Display': 0.10, 'Social': 0.15},
    'current_reach_frac': {'TV': 0.2, 'Display': 0.5, 'Social': 0.95},
    'seasonal_amplitude': {'TV': 0.6, 'Display': 0.3, 'Social': 0.1},
    'geo_audience_heterogeneity_sd': 0.25,
    'frequency_range': {'TV': (1.0, 6.0), 'Display': (1.0, 6.0), 'Social': (1.0, 6.0)},
    'target_roi': {'TV': 3.5, 'Display': 2.0, 'Social': 1.5},
}

config = SimulationConfig.from_dict(BASE_CONFIG_OVERRIDES)
config

In [ ]:
sim, data, ground_truth = build_simulated_input(config)
sim.plot_media_time_series()
plt.show()

print(f'true ec_m:  {sim.ec_m.numpy()}')
print(f'true roi_m: {ground_truth["roi_m"]}')

## 2. Single-seed illustration: default vs. `ec_only`

Two `ModelSpec` configurations, fit on the *same* simulated dataset, so any
difference in recovery is attributable to the `ec_m` prior alone --
`alpha_m` is left at Meridian's default in both:

- **`default`**: no `prior=` override. `ec_m ~ TruncatedNormal(0.8, 0.8,
  [0.1, 10])`, identical for every channel.
- **`ec_only`**: a `LogNormal` `ec_m` prior centered on the reach-based
  half-saturation point -- "half of the channel's target audience reached
  at its typical frequency" -- computed the same way
  `simulate_adstock_hill_params()` derives the ground truth: from audience
  size, frequency, and population, never from the fitted posterior. In
  practice this is exactly the information a media planner already has
  (target audience, planned frequency) before any model is fit.

### Fit both variants on the same dataset

In [ ]:
# model_utils.PRIOR_VARIANTS also has a third `ec_noisy` variant (used by
# the TV-under-reach companion notebook's robustness section) -- restrict
# to the two this notebook's narrow hypothesis is actually about.
MAIN_VARIANTS = ['default', 'ec_only']
MCMC_KWARGS_SECTION2 = dict(n_chains=2, n_adapt=2000, n_burnin=500, n_keep=1000, seed=1)

summaries = {}
for variant in MAIN_VARIANTS:
  model_spec = build_model_spec(variant, sim, config)
  mmm = model.Meridian(input_data=data, model_spec=model_spec)
  mmm.sample_prior(500)
  mmm.sample_posterior(**MCMC_KWARGS_SECTION2)
  summaries[variant] = az.summary(
      mmm.inference_data, extend=True, hdi_prob=0.9
  ).reset_index()

  print(f'--- {variant} ---')
  display(filter_param_summary(summaries[variant], 'ec_m'))
  display(filter_param_summary(summaries[variant], 'roi_m'))

In [ ]:
def side_by_side(prefix, true_values, channel_names):
  rows = []
  for label, summary_df in summaries.items():
    sub = filter_param_summary(summary_df, prefix).copy()
    sub['prior'] = label
    rows.append(sub)
  out = pd.concat(rows, ignore_index=True)
  out['channel'] = out['index'].str.extract(r'\[(.*)\]')
  out['true_value'] = out['channel'].map(dict(zip(channel_names, true_values)))
  out['true_in_90pct_hdi'] = (out['true_value'] >= out['hdi_5%']) & (
      out['true_value'] <= out['hdi_95%']
  )
  return out[
      ['prior', 'channel', 'mean', 'hdi_5%', 'hdi_95%', 'true_value', 'true_in_90pct_hdi']
  ].sort_values(['channel', 'prior'])


print('ec_m: default vs. ec_only')
display(side_by_side('ec_m', sim.ec_m.numpy(), config.channel_names))
print('roi_m: default vs. ec_only')
display(side_by_side('roi_m', ground_truth['roi_m'], config.channel_names))

## 3. Multi-seed calibration study: 90% HDI coverage

A single seed is an anecdote. Here we regenerate the dataset from scratch
under ~20 different random seeds (same `BASE_CONFIG_OVERRIDES` each time),
fit both `default` and `ec_only` on each, and record whether the true
`ec_m` (and `roi_m`) falls inside the resulting 90% HDI. **Note:** to keep
~20+ reps practical, MCMC settings below (`n_adapt`/`n_burnin`/`n_keep`)
are reduced from Section 2's -- this trades some per-fit precision for
enough repetitions to estimate coverage. Increase `N_SEEDS` and the MCMC
settings for a more rigorous (and much slower) run.

In [ ]:
N_SEEDS = 20
SEED_NUMS = list(range(1000, 1000 + N_SEEDS))

# Reduced relative to Section 2 -- see markdown note above.
MCMC_KWARGS = dict(n_chains=2, n_adapt=500, n_burnin=200, n_keep=250, seed=1)
HDI_PROB = 0.9
PARAM_PREFIXES = ('ec_m', 'roi_m')

In [ ]:
def true_values_for_seed(sim, ground_truth):
  return {
      'ec_m': dict(zip(config.channel_names, sim.ec_m.numpy())),
      'roi_m': dict(zip(config.channel_names, ground_truth['roi_m'])),
  }


def coverage_rows_for_fit(summary_df, prior_label, seed_num, truth):
  rows = []
  hdi_lo_col = f'hdi_{round((1 - HDI_PROB) / 2 * 100)}%'
  hdi_hi_col = f'hdi_{round((1 - (1 - HDI_PROB) / 2) * 100)}%'
  for prefix in PARAM_PREFIXES:
    sub = filter_param_summary(summary_df, prefix)
    for _, row in sub.iterrows():
      channel = row['index'].split('[')[-1].rstrip(']')
      if channel not in truth[prefix]:
        continue
      true_value = truth[prefix][channel]
      rows.append({
          'seed_num': seed_num,
          'prior': prior_label,
          'param': prefix,
          'channel': channel,
          'mean': row['mean'],
          'true_value': true_value,
          'covered': bool(
              row[hdi_lo_col] <= true_value <= row[hdi_hi_col]
          ),
      })
  return rows

In [ ]:
coverage_rows = []
seed_variant_failures = []

for seed_num in SEED_NUMS:
  seed_overrides = dict(BASE_CONFIG_OVERRIDES, seed_num=seed_num)
  seed_config = SimulationConfig.from_dict(seed_overrides)
  try:
    seed_sim, seed_data, seed_ground_truth = build_simulated_input(seed_config)
  except Exception as e:  # pylint: disable=broad-except
    seed_variant_failures.append((seed_num, 'ALL', repr(e)))
    print(f'seed {seed_num}: data generation FAILED ({e!r}) -- all variants skipped')
    continue

  truth = true_values_for_seed(seed_sim, seed_ground_truth)
  n_ok = 0
  for variant in MAIN_VARIANTS:
    try:
      seed_model_spec = build_model_spec(variant, seed_sim, seed_config)
      seed_mmm = model.Meridian(input_data=seed_data, model_spec=seed_model_spec)
      seed_mmm.sample_prior(200)
      seed_mmm.sample_posterior(**MCMC_KWARGS)
      seed_summary = az.summary(
          seed_mmm.inference_data, hdi_prob=HDI_PROB
      ).reset_index()
      coverage_rows += coverage_rows_for_fit(seed_summary, variant, seed_num, truth)
      n_ok += 1
    except Exception as e:  # pylint: disable=broad-except
      seed_variant_failures.append((seed_num, variant, repr(e)))
      print(f'seed {seed_num} [{variant}]: FAILED ({e!r}) -- skipped')

  print(f'seed {seed_num}: done ({n_ok}/{len(MAIN_VARIANTS)} variants succeeded)')

n_expected = len(SEED_NUMS) * len(MAIN_VARIANTS)
print(f'\n{n_expected - len(seed_variant_failures)}/{n_expected} seed x variant fits'
      f' completed successfully ({len(seed_variant_failures)} failures -- see'
      ' seed_variant_failures for detail)')
coverage_df = pd.DataFrame(coverage_rows)

### Results: 90% HDI coverage rate, default vs. `ec_only`

A well-calibrated 90% HDI should contain the true value roughly 90% of the
time. Coverage well below that indicates the prior/model combination is
systematically failing to recover the true parameter -- not just adding
noise, but being confidently wrong.

In [ ]:
coverage_summary = (
    coverage_df.groupby(['param', 'channel', 'prior'])['covered']
    .mean()
    .unstack('prior')
    .reindex(columns=MAIN_VARIANTS)
    .rename(columns=lambda c: f'{c}_coverage')
)
coverage_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = coverage_summary.reset_index()
labels = plot_df['param'] + '[' + plot_df['channel'] + ']'
x = np.arange(len(plot_df))
variant_names = MAIN_VARIANTS
n_variants = len(variant_names)
width = 0.8 / n_variants

for i, variant in enumerate(variant_names):
  offset = (i - (n_variants - 1) / 2) * width
  ax.bar(x + offset, plot_df[f'{variant}_coverage'], width, label=variant)

ax.axhline(0.9, color='black', linestyle='--', linewidth=1, label='nominal 90%')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('90% HDI coverage rate')
n_completed_seeds = len(SEED_NUMS) - len({s for s, v, _ in seed_variant_failures if v == 'ALL'})
ax.set_title(f'HDI coverage across {n_completed_seeds} seeds: '
             f'{", ".join(variant_names)} priors')
ax.legend()
plt.tight_layout()
plt.show()

## Conclusion

If the bars above show `default` `ec_m` coverage well below the nominal
90% line (and, likely to a lesser degree since it's a downstream function
of other parameters too, `roi_m` as well) while `ec_only` sits close to
90%, that's the calibration evidence for this narrow hypothesis:
**Meridian's default `ec_m` prior is not merely uninformative -- on data
where the truth is known, it produces credible intervals that under-cover
the true half-saturation point in a systematic, repeatable way, and a
reach-based `ec_m` prior -- derivable from ordinary media-planning inputs,
not from the fitted posterior -- restores that coverage.**